In [1]:
import sys
import os
sys.path.append(os.path.abspath('..')) # Points Python to the parent folder

from data.market_data import MarketDataFetcher
import datetime
import pandas as pd 
import numpy as np

start_date = datetime.date(2021, 1, 1)
end_date = datetime.date.today()
tickers = ["msft", "aapl"]
md = MarketDataFetcher()

df = md.fetch_multiple(tickers=tickers, start_date=start_date, end_date= end_date)

close_prices = pd.DataFrame({
    ticker: data['Close'].squeeze() 
    for ticker, data in df.items()
})


close_prices["msft"]

Date
2021-01-04    207.956146
2021-01-05    208.156738
2021-01-06    202.759369
2021-01-07    208.529266
2021-01-08    209.799820
                 ...    
2026-07-20    402.290009
2026-07-21    397.750000
2026-07-22    390.339996
2026-07-23    381.579987
2026-07-24           NaN
Name: msft, Length: 1395, dtype: float64

In [2]:
cov_matrix = close_prices.cov()
cov_matrix.iloc[0, 1]

3473.6502356901874

In [3]:
corr = close_prices.corr()
corr.iloc[0, 1]

0.8160596085567823

In [4]:
x_raw = np.array([0.01, -0.02, 0.03, 0.00, -0.01]) 
x_raw

array([ 0.01, -0.02,  0.03,  0.  , -0.01])

In [5]:
ones = np.ones(len(x_raw))
ones

array([1., 1., 1., 1., 1.])

In [6]:
stacked = np.vstack([x_raw, ones])
stacked

array([[ 0.01, -0.02,  0.03,  0.  , -0.01],
       [ 1.  ,  1.  ,  1.  ,  1.  ,  1.  ]])

In [7]:
X_final = stacked.T
X_final

array([[ 0.01,  1.  ],
       [-0.02,  1.  ],
       [ 0.03,  1.  ],
       [ 0.  ,  1.  ],
       [-0.01,  1.  ]])

In [8]:
df_benchmark = md.get_historical_prices("^SPX", start_date="2025-01-01", end_date="2026-01-01")
df_benchmark["Close"]

Ticker,^SPX
Date,
2025-01-02,5868.549805
2025-01-03,5942.470215
2025-01-06,5975.379883
2025-01-07,5909.029785
2025-01-08,5918.250000
...,...
2025-12-24,6932.049805
2025-12-26,6929.939941
2025-12-29,6905.740234


In [9]:
import sys
import os
sys.path.append(os.path.abspath('..')) # Points Python to the parent folder
from domain.transaction import Transaction
from domain.position import Position
from domain.portfolio import Portfolio
from data.market_data import MarketDataFetcher
from analytics.timeseries import PortfolioTimeSeries
from analytics.performance import PerformanceCalculator
from analytics.twr import TWRcalculator
from analytics.mwr import MWRCalculator
from datetime import datetime
import pandas as pd
from reporting.summary import PortfolioSummary


In [10]:
transaction1 = Transaction('aapl', '2024-01-02','BUY', 5, 185, 2,'USD')
transaction2 = Transaction('msft', '2024-03-01','BUY', 3, 415, 2,'USD')
transaction3 = Transaction('aapl', '2026-06-01','BUY', 1, 195, 2,'USD')
transaction4 = Transaction('amzn', '2026-03-01','BUY', 1, 120, 2,'USD')

In [11]:
port1 = Portfolio("ATK_1", "USD", creation_date = '2024-01-01')
port1

In [12]:
port1.add_transaction(transaction1)
port1.add_transaction(transaction2)
port1.add_transaction(transaction3)
port1.add_transaction(transaction4)

In [13]:
fetcher = MarketDataFetcher()
time_s = PortfolioTimeSeries(port1, fetcher=fetcher, start_date=None)

In [22]:

portfolio_val = time_s.portfolio_value()
portfolio_ret = portfolio_val.pct_change().dropna()
portfolio_ret

2024-01-03   -0.007488
2024-01-04   -0.012700
2024-01-05   -0.004013
2024-01-08    0.024175
2024-01-09   -0.002263
                ...   
2026-07-22   -0.010548
2026-07-23   -0.016524
2026-07-24    0.000000
2026-07-27    0.000000
2026-07-28    0.000000
Freq: B, Length: 670, dtype: float64

In [21]:
pc = PerformanceCalculator(portfolio_value = portfolio_val, portfolio_returns = portfolio_ret, risk_free= 0.01)

In [23]:
total_return = pc.total_return()
total_return

2.350036193055794

In [17]:
cash_flow_mwr = time_s.mwr_cashflows
today_port_worth= portfolio_val.iloc[-1]
inception_date = pd.to_datetime(port1.creation_date)


mwr_calc=MWRCalculator(cashflows=cash_flow_mwr,
                       current_value=today_port_worth,
                       inception_date=inception_date )

mwr_df = mwr_calc.calculate_mwr()
mwr_df

0.09925749682081661

In [18]:
c_dates = time_s.cashflow_dates
positions = port1.open_positions()
twr_calc = TWRcalculator(portfolio_value= portfolio_val,positions=positions)
df_twr = twr_calc.calculate_twr()
df_twr

0.23179265880917344